In [ ]:
import os
import subprocess

import pandas as pd
import pyspark
from pyspark.sql import SparkSession

In [ ]:
pyspark.__version__

In [ ]:
path = './data'
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet'
output_path = 'data/yellow_tripdata_2025-11.parquet'

if os.path.isdir(path) and os.path.exists(output_path) == False:
    result = subprocess.run(["wget", "-O", output_path, url])
elif os.path.exists(output_path) == True:
    print('File already exists.')
else:
    os.mkdir(path)
    result = subprocess.run(["wget", "-O", output_path, url])

In [ ]:
!$JAVA_HOME/bin/java -version

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework_retry') \
    .getOrCreate()

In [ ]:
df = spark.read.parquet('./data/yellow_tripdata_2025-11.parquet')

In [ ]:
df.columns

In [ ]:
df.show()

In [ ]:
df = df.repartition(4)

In [ ]:
df.write.parquet('data/2025/11/')

In [ ]:
path = './data/2025/11'
df_data = spark.read.parquet(path)

In [ ]:
df_data.createOrReplaceTempView('df_data')

In [ ]:
# Number of taxi trips on Nov-15 (row counts)
spark.sql("""
SELECT
    count(*)
FROM
    df_data
WHERE 
    to_date(tpep_pickup_datetime) = '2025-11-15'
""").show()

In [ ]:
df_with_hours = spark.sql("""
    SELECT 
        *,
        (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS duration_hours
    FROM df_data
    ORDER BY duration_hours DESC
""").show()

In [ ]:
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_output_path = 'data/zone_lookup.csv'

result = subprocess.run(["wget", "-O", zone_output_path, zone_url])

In [ ]:
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("./data/zone_lookup.csv")

In [ ]:
# df_trips es tu DataFrame de viajes de taxi
df_with_zones = df_data.join(df_zones, df_data.PULocationID == df_zones.LocationID, how='left')

In [ ]:
df_with_zones.show()

In [ ]:
df_with_zones.createOrReplaceTempView("trips_with_zones")

In [ ]:
least_frecuent_zone = spark.sql("""
    SELECT 
        Zone, 
        COUNT(*) as total_trips
    FROM trips_with_zones
    GROUP BY Zone
    ORDER BY total_trips ASC
    LIMIT 5
""")

least_frecuent_zone.show()